In [ ]:
"""
build_adr_ml_dataset_prod.py
Production-ready fusion: MIMIC outputs + FAERS -> ML-ready X_features.csv + y_target.csv
OPTIMIZED FOR LOW MEMORY (STREAM-TO-DISK)

Outputs:
  output/ml_ready/X_features.csv
  output/ml_ready/y_target.csv
  output/ml_ready/feature_manifest.json
  output/ml_ready/encoders.json

Notes:
 - Runs in 2 passes to handle large data without RAM crashes.
 - Pass 1: Scan for vocabulary (Enums/Encoders)
 - Pass 2: Process chunks, merge with in-memory auxiliary tables, write to disk.
"""

import json
import warnings
import gc
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from tqdm import tqdm

warnings.filterwarnings("ignore")

# ---- Google Colab Paths (Hardcoded) ----
DATA_DIR = Path("/content/drive/MyDrive/medbot/colabupload")
OUT_DIR = Path("/content/drive/MyDrive/medbot/colabupload/output/ml_ready")

# Ensure output directory exists
OUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"INFO: Running in Colab-Only Mode (Memory Optimized)")
print(f"INFO: Using DATA_DIR={DATA_DIR}")
print(f"INFO: Using OUT_DIR={OUT_DIR}")

# ---- Configurable parameters ----
CHUNK_SIZE = 50000  # Process prescriptions in chunks of 50k rows
FAERS_HIGH_SEVERE_THRESHOLD = 0.05
WEAK_SUPERVISION_SCORE_THRESHOLD = 4

# ---- Utility helpers ----
def check_file(path: Path):
    if not path.exists():
        print(f"WARNING: missing file: {path}")
        return False
    return True

def normalize_drug(name):
    if pd.isna(name):
        return ""
    return (
        str(name)
        .lower()
        .strip()
        .replace("(", "")
        .replace(")", "")
        .replace("-", " ")
    )

def safe_read_csv(path: Path, **kwargs):
    if not check_file(path):
        return None
    return pd.read_csv(path, **kwargs)


# ---- 1. Load & Compact Auxiliary Tables (In-Memory) ----
def load_and_compact_auxiliary():
    print("INFO: Loading and compacting auxiliary tables...")
    aux = {}

    # 1. FAERS (Small)
    faers_path = DATA_DIR / "faers_drug_summary.csv"
    aux["faers"] = safe_read_csv(faers_path)

    # 2. Admissions (Medium - Keep all cols for now, assuming 1 row per admission)
    adm_path = DATA_DIR / "mimic_admission_summary.csv"
    aux["admissions"] = safe_read_csv(adm_path)

    # 3. ICU Stays (Medium - needed for LOS and stay_id mapping)
    icu_path = DATA_DIR / "mimic_icustays.csv"
    icustays = safe_read_csv(icu_path)
    aux["icustays"] = icustays

    # 4. Labs (Usually Large - Pivot and Compact immediately)
    lab_path = DATA_DIR / "mimic_lab_summary.csv"
    lab_df = safe_read_csv(lab_path)
    aux["lab_block"] = build_lab_block(lab_df)
    del lab_df # free raw
    gc.collect()

    # 5. Vitals (Large - Pivot and Compact immediately)
    vital_path = DATA_DIR / "mimic_vital_signs_summary.csv"
    vital_df = safe_read_csv(vital_path)
    aux["vital_block"] = build_vital_block(vital_df, icustays)
    del vital_df # free raw
    gc.collect()

    # 6. Flags (Small)
    aux["dialysis_flags"] = safe_read_csv(DATA_DIR / "dialysis_flags.csv")
    aux["pressor_flags"] = safe_read_csv(DATA_DIR / "pressor_flags.csv")

    print("INFO: Auxiliary tables loaded and compacted.")
    return aux

def scan_aux_vocab(aux):
    """Scan in-memory aux tables for object columns to encode (e.g. Gender, Ethnicity)."""
    print("INFO: Scanning Auxiliary Tables for generic vocab...")
    new_encoders = {}
    ignore_cols = ["subject_id", "hadm_id", "starttime", "drug", "drug_norm", "stay_id"]

    for key, df in aux.items():
        if isinstance(df, pd.DataFrame):
            for col in df.select_dtypes(include=["object", "category"]).columns:
                 if col in ignore_cols: continue
                 if col in new_encoders: continue # already found

                 print(f"INFO: Found text column '{col}' in {key}. Building encoder.")
                 le = LabelEncoder()
                 # Ensure strings
                 vals = df[col].astype(str).fillna("<<NA>>").unique()
                 le.fit(list(vals) + ["<<NA>>"])
                 new_encoders[col] = le
    return new_encoders

# ---- Logic reused from v1 (modified for standalone usage) ----

def build_faers_lookup(faers_df: pd.DataFrame):
    if faers_df is None or faers_df.empty:
        return {}
    faers_df = faers_df.copy()
    if "ADR_Rate" not in faers_df.columns: faers_df["ADR_Rate"] = 0.0
    if "Severe_Outcome_Rate" not in faers_df.columns: faers_df["Severe_Outcome_Rate"] = 0.0

    faers_df["drug_norm"] = faers_df["drugname"].apply(normalize_drug)
    # create dict
    lookup = {}
    for _, r in faers_df.iterrows():
        lookup[r["drug_norm"]] = {
            "ADR_Rate": float(r.get("ADR_Rate", 0.0) or 0.0),
            "Severe_Outcome_Rate": float(r.get("Severe_Outcome_Rate", 0.0) or 0.0),
        }
    return lookup

def build_drug_block_chunk(rx_chunk: pd.DataFrame, faers_lookup: dict):
    # Process a single chunk of prescriptions
    rx = rx_chunk.copy()
    for col in ["subject_id", "hadm_id"]:
        if col not in rx.columns: rx[col] = np.nan

    drug_col = "drug" if "drug" in rx.columns else ("drug_name" if "drug_name" in rx.columns else None)
    if drug_col is None:
        rx["drug"] = ""
        drug_col = "drug"

    rx["drug_norm"] = rx[drug_col].apply(normalize_drug)

    # Apply FAERS lookup
    def get_faers_stats(d_norm):
        entry = faers_lookup.get(d_norm, {})
        return entry.get("ADR_Rate", 0.0), entry.get("Severe_Outcome_Rate", 0.0)

    # Vectorized application might be faster but apply is safe enough for chunks
    stats = rx["drug_norm"].apply(get_faers_stats)
    rx["faers_adr_rate"] = [x[0] for x in stats]
    rx["faers_severe_rate"] = [x[1] for x in stats]
    rx["high_risk_drug"] = (rx["faers_severe_rate"] >= FAERS_HIGH_SEVERE_THRESHOLD).astype(int)

    # Duration
    if "duration_hours" in rx.columns:
        rx["duration_days"] = rx["duration_hours"] / 24.0
    elif "duration" in rx.columns:
        rx["duration_days"] = rx["duration"]
    else:
        rx["duration_days"] = np.nan

    dose_col = next((c for c in ["dose_val_rx", "dose", "dose_val"] if c in rx.columns), None)
    route_col = next((c for c in ["route", "admin_route"] if c in rx.columns), None)

    out = rx[["subject_id", "hadm_id", drug_col]].rename(columns={drug_col: "drug"})
    out["drug_norm"] = rx["drug_norm"]
    out["dose_val_rx"] = rx[dose_col] if dose_col is not None else np.nan
    out["route"] = rx[route_col] if route_col is not None else np.nan
    out["duration_days"] = rx["duration_days"]
    out["faers_adr_rate"] = rx["faers_adr_rate"]
    out["faers_severe_rate"] = rx["faers_severe_rate"]
    out["high_risk_drug"] = rx["high_risk_drug"]

    return out

def build_lab_block(lab_summary_df: pd.DataFrame):
    if lab_summary_df is None: return pd.DataFrame(columns=["hadm_id"])
    df = lab_summary_df.copy()

    # Attempt Pivot
    if {"hadm_id", "lab_name", "last_value"}.issubset(df.columns):
        pivot = df.pivot_table(index="hadm_id", columns="lab_name", values="last_value", aggfunc="last").reset_index()
    elif {"hadm_id", "lab_name", "first_value", "last_value"}.issubset(df.columns):
        pivot = df.pivot_table(index="hadm_id", columns="lab_name", values="last_value", aggfunc="last").reset_index()
    elif "hadm_id" in df.columns and any(col.startswith("lab_") for col in df.columns):
        pivot = df.copy() # Already pivoted
    else:
        return pd.DataFrame(columns=["hadm_id"])

    # Clean cols
    cols = pivot.columns.tolist()
    renamed = ["hadm_id"] + [f"lab_{str(c).lower().replace(' ', '_')}" for c in cols[1:]]
    pivot.columns = renamed
    return pivot

def build_vital_block(vital_summary_df: pd.DataFrame, icustays_df: pd.DataFrame):
    if vital_summary_df is None: return pd.DataFrame(columns=["hadm_id"])
    vs = vital_summary_df.copy()

    # Normalize col names
    if {"stay_id", "vital_sign", "mean"}.issubset(vs.columns):
        vs = vs.rename(columns={"vital_sign": "vital", "mean": "mean_val"})
    elif {"stay_id", "vital", "mean"}.issubset(vs.columns):
        vs = vs.rename(columns={"mean": "mean_val"})

    # Pivot
    if {"stay_id", "vital", "mean_val"}.issubset(vs.columns):
        pivot = vs.pivot_table(index="stay_id", columns="vital", values="mean_val", aggfunc="mean").reset_index()
        pivot.columns = ["stay_id"] + [f"vital_{str(c).lower().replace(' ', '_')}" for c in pivot.columns[1:]]
    else:
        pivot = vs.copy()

    if icustays_df is None or "stay_id" not in pivot.columns:
        return pd.DataFrame(columns=["hadm_id"])

    # Map stay -> hadm
    icu_map = icustays_df[["stay_id", "hadm_id"]].drop_duplicates()
    pivot = pivot.merge(icu_map, on="stay_id", how="left")

    # Aggregate to hadm
    agg_cols = [c for c in pivot.columns if c not in ("stay_id", "hadm_id")]
    hadm_vitals = pivot.groupby("hadm_id")[agg_cols].mean().reset_index()
    hadm_vitals.columns = ["hadm_id"] + [f"{c}" for c in hadm_vitals.columns.tolist()[1:]]
    return hadm_vitals

def weak_supervision_labeling(df):
    score = np.zeros(len(df), dtype=float)
    score += (df.get("high_risk_drug", 0) == 1).astype(int) * 2
    score += (df.get("on_dialysis", False) == True).astype(int) * 2
    score += (df.get("aki", False) == True).astype(int) * 2
    score += (df.get("chronic_liver_disease", False) == True).astype(int) * 1

    df["weak_score"] = score
    df["ADR_flag"] = (df["weak_score"] >= WEAK_SUPERVISION_SCORE_THRESHOLD).astype(int)
    return df

# ---- Main Stream Processing ----

def run_pass_1_vocab(rx_path: Path):
    """Scan prescriptions to find all unique drugs/routes for Encoder fitting."""
    print("INFO: Pass 1 - Building Vocab (Encoders)...")
    if not rx_path.exists():
        return None

    unique_vals = {"drug": set(), "route": set()}

    for chunk in tqdm(pd.read_csv(rx_path, chunksize=CHUNK_SIZE), desc="Scanning Vocab"):
        # Normalize drug info
        drug_col = "drug" if "drug" in chunk.columns else ("drug_name" if "drug_name" in chunk.columns else None)
        if drug_col:
            # We encode the NORMALIZED drug name usually, or raw?
            # Original code encoded all object columns. Drug name is likely object.
            # Let's normalize first to match build_drug_block logic
            chunk["drug_temp"] = chunk[drug_col].apply(normalize_drug)
            unique_vals["drug"].update(chunk["drug_temp"].dropna().unique())

        route_col = next((c for c in ["route", "admin_route"] if c in chunk.columns), None)
        if route_col:
            chunk[route_col] = chunk[route_col].astype(str).fillna("<<NA>>")
            unique_vals["route"].update(chunk[route_col].unique())

    # Build Encoders
    encoders = {}
    for col, vals in unique_vals.items():
        le = LabelEncoder()
        le.fit(list(vals) + ["<<NA>>"]) # Ensure NA is known
        encoders[col] = le

    print("INFO: Pass 1 Complete. Encoders built.")
    return encoders

def run_pass_2_stream(rx_path: Path, aux: dict, encoders: dict):
    """Process chunks, merge, encode, and write to disk."""
    print("INFO: Pass 2 - Stream Processing & Writing...")

    faers_lookup = build_faers_lookup(aux.get("faers"))

    first_chunk = True
    total_processed = 0

    X_out_path = OUT_DIR / "X_features.csv"
    y_out_path = OUT_DIR / "y_target.csv"

    # Remove existing files if any
    if X_out_path.exists(): X_out_path.unlink()
    if y_out_path.exists(): y_out_path.unlink()

    for chunk in tqdm(pd.read_csv(rx_path, chunksize=CHUNK_SIZE), desc="Processing Chunks"):
        # 1. Build Drug Block (Base)
        drug_df = build_drug_block_chunk(chunk, faers_lookup)

        # 2. Merge Admissions (Left Join)
        if aux.get("admissions") is not None:
             merged = drug_df.merge(aux["admissions"], on=["subject_id", "hadm_id"], how="left", suffixes=("", "_adm"))
        else:
             merged = drug_df

        # 3. Merge Labs (Left Join)
        if aux.get("lab_block") is not None:
            merged = merged.merge(aux["lab_block"], on="hadm_id", how="left")

        # 4. Merge Vitals (Left Join)
        if aux.get("vital_block") is not None:
            merged = merged.merge(aux["vital_block"], on="hadm_id", how="left")

        # 5. Merge ICU LOS
        if aux.get("icustays") is not None:
            icu_agg = aux["icustays"].groupby("hadm_id")["los"].sum().reset_index().rename(columns={"los":"icu_total_los"})
            merged = merged.merge(icu_agg, on="hadm_id", how="left")

        # 6. Flags
        if aux.get("dialysis_flags") is not None and "hadm_id" in aux["dialysis_flags"].columns:
            merged = merged.merge(aux["dialysis_flags"], on="hadm_id", how="left")
        else:
            merged["on_dialysis"] = merged.get("on_dialysis", False)

        if aux.get("pressor_flags") is not None and "hadm_id" in aux["pressor_flags"].columns:
            merged = merged.merge(aux["pressor_flags"], on="hadm_id", how="left")
        else:
            merged["on_vasopressors"] = merged.get("on_vasopressors", False)

        # 7. Fill NaNs (General)
        for col in ["aki","chronic_liver_disease","hypertension","diabetes_type1","diabetes_type2","ckd"]:
            if col not in merged.columns: merged[col] = False

        num_cols = merged.select_dtypes(include=[np.number]).columns.tolist()
        merged[num_cols] = merged[num_cols].fillna(0)
        bool_cols = merged.select_dtypes(include=["bool"]).columns.tolist()
        merged[bool_cols] = merged[bool_cols].fillna(False)

        # 8. Create Target (Weak Supervision)
        labeled_df = weak_supervision_labeling(merged)

        # 9. Encoding & Finalizing
        y_chunk = labeled_df["ADR_flag"]

        drop_cols = ["ADR_flag", "drug", "drug_norm", "starttime", "subject_id", "hadm_id", "route"]
        # Note: 'route' and 'drug_norm' acts as our categorical sources

        X_chunk = labeled_df.copy()

        # Apply generic encoders (including drug/route if they were found in pass 1/2)
        # We manually attach drug/route encoders in pass 1.
        # Check standard encoders
        for col, le in encoders.items():
            if col == "drug":
                X_chunk["drug_encoded"] = le.transform(X_chunk["drug_norm"].fillna("<<NA>>"))
            elif col == "route":
                X_chunk["route_encoded"] = le.transform(X_chunk["route"].astype(str).fillna("<<NA>>"))
            elif col in X_chunk.columns:
                # Generic column found in merged (e.g. Gender from admissions)
                X_chunk[col] = le.transform(X_chunk[col].astype(str).fillna("<<NA>>"))

        # Now select final numerical features
        # Filter all object columns that are NOT in encoders (we can't use them)
        # Select numeric columns
        final_cols = []
        for c in X_chunk.columns:
            if c in drop_cols: continue
            if c == "drug_temp": continue
            # If numeric, keep
            if pd.api.types.is_numeric_dtype(X_chunk[c]):
                final_cols.append(c)

        X_final = X_chunk[final_cols]
        # Fill any remaining NaNs
        X_final = X_final.fillna(0)

        # Write to Disk
        mode = "w" if first_chunk else "a"
        header = True if first_chunk else False

        X_final.to_csv(X_out_path, mode=mode, header=header, index=False)
        y_chunk.to_csv(y_out_path, mode=mode, header=header, index=False)

        # Capture columns for manifest on first chunk
        if first_chunk:
            manifest = {"features": X_final.columns.tolist()}
            with open(OUT_DIR / "feature_manifest.json", "w") as f:
                json.dump(manifest, f, indent=2)

        first_chunk = False
        total_processed += len(X_final)

        # Cleanup
        del chunk, drug_df, merged, labeled_df, X_chunk, X_final, y_chunk
        gc.collect()

    print(f"✓ Pass 2 Complete. Total Rows: {total_processed}")

    # Save Encoders classes
    encoder_data = {col: le.classes_.tolist() for col, le in encoders.items()}
    with open(OUT_DIR / "encoders.json", "w") as f:
        json.dump(encoder_data, f, indent=2)
    print(f"✓ Saved encoders -> {OUT_DIR / 'encoders.json'}")

def run_all_optimized():
    rx_path = DATA_DIR / "mimic_prescriptions.csv"
    if not rx_path.exists():
        print("CRITICAL: Prescriptions file missing. Cannot proceed.")
        return

    # 1. Load Aux
    aux = load_and_compact_auxiliary()

    # 2. Pass 1 (Encoders)
    encoders = run_pass_1_vocab(rx_path)

    # NEW: Scan aux for extra string columns
    extra_encoders = scan_aux_vocab(aux)
    encoders.update(extra_encoders)

    # 3. Pass 2 (Stream)
    run_pass_2_stream(rx_path, aux, encoders)

    print("DONE. Stream processing finished.")

if __name__ == "__main__":
    run_all_optimized()


INFO: Running in Colab-Only Mode (Memory Optimized)
INFO: Using DATA_DIR=/content/drive/MyDrive/medbot/colabupload
INFO: Using OUT_DIR=/content/drive/MyDrive/medbot/colabupload/output/ml_ready
INFO: Loading and compacting auxiliary tables...
INFO: Auxiliary tables loaded and compacted.
INFO: Pass 1 - Building Vocab (Encoders)...


Scanning Vocab: 406it [01:30,  4.48it/s]


INFO: Pass 1 Complete. Encoders built.
INFO: Scanning Auxiliary Tables for generic vocab...
INFO: Found text column 'drugname' in faers. Building encoder.
INFO: Found text column 'indi_pt' in faers. Building encoder.
INFO: Found text column 'admittime' in admissions. Building encoder.
INFO: Found text column 'dischtime' in admissions. Building encoder.
INFO: Found text column 'deathtime' in admissions. Building encoder.
INFO: Found text column 'admission_type' in admissions. Building encoder.
INFO: Found text column 'admit_provider_id' in admissions. Building encoder.
INFO: Found text column 'admission_location' in admissions. Building encoder.
INFO: Found text column 'discharge_location' in admissions. Building encoder.
INFO: Found text column 'insurance' in admissions. Building encoder.
INFO: Found text column 'language' in admissions. Building encoder.
INFO: Found text column 'marital_status' in admissions. Building encoder.
INFO: Found text column 'race' in admissions. Building enc

Processing Chunks: 406it [32:01,  4.73s/it]


✓ Pass 2 Complete. Total Rows: 20292611
✓ Saved encoders -> /content/drive/MyDrive/medbot/colabupload/output/ml_ready/encoders.json
DONE. Stream processing finished.


Once your Google Drive is mounted, I will modify the `OUT_DIR` variable in the `build_adr_ml_dataset_prod.py` script to save the files to `/content/drive/MyDrive/colab_output`.